# Traditional RAG over the *12 Angry Men* Screenplay

**Author:** Giannis Mertziotis · **Co-author:** Spyridon Tzokas

Part 2 of a two-part AI lab project (NTUA, ECE). This notebook builds a **traditional Retrieval-Augmented Generation (RAG)** system over the screenplay of *12 Angry Men* and evaluates it on a shared question set.

**Approach.** The script PDF is downloaded and parsed (PyMuPDF), split into chunks, and indexed in two ways: a **keyword** index (BM25) and a **vector** index (FAISS over sentence-transformer embeddings). At query time both indexes are searched, the retrieved chunks are passed as context to **Qwen3-4B-Instruct**, and the generated answers are written to a CSV for evaluation.

> Note: the screenplay text is **not** committed to this repo for copyright reasons — the notebook downloads it at runtime.


# Visualize the dataset

To start with, upload the dataset in Google Colab. Then you can print the first lines to make sure it is correct.

In [7]:
import pandas as pd

# Load the dataset
df = pd.read_csv("dataset.csv")

display(df.head())

,Question,Correct Answer,Page References,Explanation
0,What type of train passes by the apartment rig...,An el train (elevated train).,26,This requires retrieving a specific environmen...
1,"According to the boy's alibi, where did he cla...",He claimed to be at the movies.,25,Retrieval of a specific plot point established...
2,What does the sign on the first door of the co...,Court of General Sessions. Part I,3,Fact explicitly stated in the script in the de...
3,What pattern does Juror #3 draw on the same sh...,tic-tac-toe pattern,64,Fact explicitly stated in the script. The info...
4,Which Juror says that he's lived in a slum all...,Juror #5,31,Fact explicitly stated in the script on page 31


# Building a Traditional RAG System: 12 Angry Men

In this exercise, we are going to build a basic Retrieval-Augmented Generation (RAG) system from scratch. Our target data is the script for the movie *12 Angry Men*.

A Large Language Model (LLM) alone might not know specific details about this script. To fix this, we will:
1. Download the raw PDF of the script and extract its text.
2. Break the script down into smaller pieces (chunks).
3. Store these chunks in Keyword (BM25) and Vector (FAISS) databases.
4. Search both databases when a question is asked.
5. Provide the retrieved chunks to the LLM so it can answer the question accurately.

# Step 1: Install required libraries
In this first step, we install all the necessary Python packages for building our retrieval and language model pipeline.

The command below uses `pip` to quietly (`-q`) install:

- **transformers** → for loading and using large language models (LLMs)
- **accelerate** → for efficient model execution (especially on GPUs)
- **sentence-transformers** → for generating text embeddings
- **faiss-cpu** → for fast similarity search in vector databases
- **rank_bm25** → for traditional keyword-based retrieval
- **pandas** → for handling and analyzing tabular data
- **PyMuPDF** → for reading and processing PDF documents

> Run this cell once before continuing with the rest of the notebook.


In [8]:
!pip install -q transformers accelerate sentence-transformers faiss-cpu rank_bm25 pandas PyMuPDF

## Step 2: Data Ingestion & Document Chunking

Before we can search through our movie script, we need to load the PDF and break it down into smaller, digestible pieces called **chunks**.

Why do we chunk?
1. **Context Limits:** Large Language Models (LLMs) have a maximum context window. We can't feed a 100-page PDF into the prompt all at once.
2. **Retrieval Accuracy:** If our search system returns massive blocks of text, the LLM might get distracted by irrelevant information ("lost in the middle" effect). Smaller, focused chunks lead to more accurate answers. How many chunks we provide the LLM is not set of course, depending on the LLM capabilities we can give smaller or larger number of chunks.


### Chunking for this assignment: Structural Chunking (Page-Based)
Instead of arbitrary word counts, we could use the document's natural structure. Of course there are other ways to create the chunks as there is no requirement for them to be of equal size.

In the cell below, we will:
1. Download the script for *12 Angry Men*.
2. Iterate through the PDF **page-by-page** using `PyMuPDF`.
3. Save each page as its own chunk.

> **Tip for RAG:** Notice how we inject `--- Page X ---` at the top of every chunk. This is a form of **metadata enrichment**. By embedding the page number directly into the text, we give the LLM the ability to cite its sources when answering the user!

In [9]:
import requests
import fitz  # PyMuPDF

# 1. Download the movie script PDF
url = "https://f004.backblazeb2.com/file/screenplays/posts/12-angry-men-1957/scripts/12%20Angry%20Men%20-%20Release.pdf"
pdf_path = "12_angry_men.pdf"

print("Downloading the script PDF...")
response = requests.get(url)
with open(pdf_path, "wb") as f:
    f.write(response.content)
print("Download complete!")

# 2. Extract text and chunk by page
print("Extracting text from PDF by page...")
doc = fitz.open(pdf_path)

chunks = []

for page_num, page in enumerate(doc):
    page_text = page.get_text().strip()

    # We only add the page if it actually contains text (skips blank pages)
    if page_text:
        # Pro-tip: Adding the page number to the chunk helps the LLM cite its sources!
        chunk_content = f"--- Page {page_num + 1} ---\n{page_text}"
        chunks.append(chunk_content)

print(f"Created {len(chunks)} chunks from {len(doc)} pages.")

# Let's peek at an example chunk
print("\n--- Example Chunk [10] (Page 11) Preview ---")
print(chunks[10][:500]) # Printing the first 500 characters of the 11th chunk
print("--------------------------------------------")

Download complete!
Extracting text from PDF by page...
Created 149 chunks from 149 pages.

--- Example Chunk [10] (Page 11) Preview ---
--- Page 11 ---
11.
#3
(smiling)
I didn't get a chance to look at the
papers today.
#4
I was just wondering how the market
closed.
#3
(pleasantly)
I wouldn't knew. Say, are you on the
exchange or something.
#4
I'm a broker.
#3
Well that's very interesting.
Listen, maybe you can answer a
question for me. I have an uncle
who’s been playing around with some
Canadian stuff...
The foreman turns around, and, as if it is an effort, calls 
out loudly to the others.
FOREMAN
All right, gentlemen. Let’s ta
--------------------------------------------


## Step 4: Building the Retrieval Databases

To enable accurate information retrieval within our RAG system, we need efficient methods to parse and search our text chunks. In this section, you will set up the core components of a hybrid search system by implementing two distinct search strategies: a sparse keyword database and a dense vector database.

*(Note: Assume the variable `chunks`, a list of text strings, is already available in your environment from the previous steps.)*

### 1. Keyword Search (BM25): Lexical Matching
BM25 operates as an advanced algorithmic version of a traditional keyword index.
* **Mechanism:** It identifies exact word matches between the user's query and the text chunks using a statistical scoring framework (based on Term Frequency-Inverse Document Frequency). It heavily weights rare, highly specific terms (e.g., "photosynthesis") while discounting common vocabulary (e.g., "the," "and").
* **Strengths:** Highly effective for retrieving specific entities, proper names, exact quotes, or strict technical jargon.
* **Limitations:** Relies strictly on lexical matching. It will fail to retrieve a chunk that exclusively uses a synonym (e.g., a query for "automobile" will not match a chunk only containing "car").

### 2. Vector Search (FAISS): Semantic Matching
Vector search identifies relevant information based on underlying meaning and context rather than exact phrasing.
* **Mechanism:** Using an embedding model, we translate text chunks into high-dimensional numerical vectors (embeddings). Texts with similar semantic meanings are positioned closer together in this mathematical space. FAISS (Facebook AI Similarity Search) rapidly calculates the spatial distance between these vectors.
* **Strengths:** Excels at understanding context and conceptual relationships (e.g., a query for "canines playing" will successfully retrieve a text chunk about "dogs fetching a ball").
* **Limitations:** Because it evaluates broad meaning, it can be overly generalized and underperform when a query requires a rigid, exact match (such as a specific user ID).

---

### Your Tasks:

**1. Keyword Database (BM25)**
* Tokenize the text `chunks` by splitting them by simple spaces.
* Initialize a `BM25Okapi` object with these tokenized chunks.

**2. Vector Database (FAISS)**
* Load the `'all-MiniLM-L6-v2'` embedding model using `SentenceTransformer`.
* Use this model to encode our `chunks` into vector embeddings.
* Find the dimensionality of these newly created embeddings.
* Create a `faiss.IndexFlatL2` index using that exact dimension.
* Add your embeddings to the FAISS index (remember to convert them to a NumPy array if they aren't already).

In [10]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print("Building Keyword Database (BM25)...")

# TODO: Tokenize 'chunks' (simple space splitting)
tokenized_chunks = [chunk.split(" ") for chunk in chunks]

# TODO: Initialize the BM25Okapi object
bm25 = BM25Okapi(tokenized_chunks)


print("Building Vector Database (FAISS)...")

# TODO: Load the 'all-MiniLM-L6-v2' embedding model
embedder =  SentenceTransformer("all-MiniLM-L6-v2")

# TODO: Convert the text chunks into vector embeddings
embeddings = ...
# convert_to_numpy=True μας δίνει κατευθείαν numpy array έτοιμο για FAISS
embeddings = embedder.encode(chunks, show_progress_bar=True, convert_to_numpy=True)

# TODO: Get the dimension of the embeddings
dimension = embeddings.shape[1]

# TODO: Create a FAISS IndexFlatL2 index using the dimension ("IndexFlatL2" function)
vector_db = vector_db = faiss.IndexFlatL2(dimension)

# TODO: Add the numpy array of embeddings to the FAISS index
vector_db.add(np.array(embeddings).astype("float32"))
print(f"FAISS index contains {vector_db.ntotal} vectors of dimension {dimension}.")

Building Keyword Database (BM25)...
Building Vector Database (FAISS)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

FAISS index contains 149 vectors of dimension 384.


## Step 5: The Hybrid Retriever

Now it is time to build the engine of our RAG system. You will write a function that takes a user's query, searches *both* of our databases, and returns the most relevant context.

Because BM25 (lexical) and FAISS (semantic) excel at different types of matching, using them together yields much better results than using either alone.

**Your Task:**
Create a `hybrid_retrieve` function that executes the following pipeline:
1. **Keyword Search:** Tokenize the incoming query, score it against your `bm25` database, and extract the indices of the top-scoring chunks.
2. **Vector Search:** Encode the query using your `embedder`, search the `vector_db`, and extract the indices of the closest vectors.
3. **The Fusion Challenge (Your Choice):** You now have two lists of "top indices." You must design a way to combine them into a single, final list of indices.
    * *Basic Approach:* Combine the lists and use a Python `set` to remove duplicates.
    * *Advanced Approach (Optional):* Implement a scoring algorithm like **Reciprocal Rank Fusion (RRF)** to mathematically weigh and re-rank the indices based on their positions in both original lists.
4. **Fetch Context:** Map your final combined indices back to the original `chunks` list to get the actual text.

### Expected Outcome
Your function should return a Python **list of text strings** (the actual text chunks). When you run the test query at the bottom of the cell, it should print the relevant text chunks that best answer the question.

In [11]:
def hybrid_retrieve(query, top_k=2):
    """
    Retrieves top chunks using BM25 and Vector Search, then combines the results.
    """

    # --- 1. Keyword Search (BM25) ---
    # TODO: Tokenize the query (split by spaces)
    tokenized_query = query.split(" ")

    # TODO: Get scores for the query from the 'bm25' object ("get_scores" function)
    bm25_scores = bm25.get_scores(tokenized_query)

    # TODO: Get the indices of the top-scoring chunks
    # argsort επιστρέφει ascending, οπότε παίρνουμε τα τελευταία top_k και τα αντιστρέφουμε
    bm25_top_indices =  bm25_top_indices = np.argsort(bm25_scores)[-top_k:][::-1].tolist()


    # --- 2. Vector Search (FAISS) ---
    # TODO: Encode the query into a vector using your 'embedder'
    query_embedding =embedder.encode([query], convert_to_numpy=True).astype("float32")


    # TODO: Search the 'vector_db' to get the top indices ("search" function )
    #επιστρέφει (distances, indices), και τα δύο 2D arrays
    distances, faiss_top_indices = vector_db.search(query_embedding, top_k)
    # Hint: FAISS returns a 2D array, you may need to extract the first element (e.g., [0].tolist())
    faiss_top_indices = faiss_top_indices[0].tolist()

    # --- 3. Combine ---
    # TODO: Implement your strategy to combine 'bm25_top_indices' and 'faiss_top_indices'.
    # You can use simple deduplication (sets), alternating selection, or advanced algorithms like RRF.
    # Ensure your final list is no longer than the desired output size (though you may fetch more initially).
    rrf_scores = {}  # dict: chunk_index → συσσωρευμένο RRF score
    k = 60

# Περνάμε τη λίστα BM25
    for rank, idx in enumerate(bm25_top_indices):
    # enumerate δίνει rank=0,1,2,... για 1η, 2η, 3η θέση
      rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k + rank)
    # .get(idx, 0): αν δεν υπάρχει ακόμα, ξεκίνα από 0

# Ίδιο και για FAISS — αν κάποιο idx υπάρχει ήδη, προστίθεται
    for rank, idx in enumerate(faiss_top_indices):
      rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k + rank)

# Ταξινόμηση με βάση τα scores, φθίνουσα
    final_combined_indices = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]

# --- 4. Fetch the Text ---
# TODO: Fetch the actual text strings from the original 'chunks' list using your final indices
    retrieved_chunks = [chunks[i] for i in final_combined_indices]

    return retrieved_chunks


# ==========================================
# You can test your hybrid retrieval function with a sample query. Adjust the query and top_k as needed to see how your fusion strategy performs.
# ==========================================
test_query = "Who had a cold?"

# Feel free to adjust top_k to see how your fusion strategy handles more chunks
results = hybrid_retrieve(test_query, top_k=2)

print(f"Found {len(results)} relevant chunks for the test query.\n")
for i, chunk in enumerate(results):
    print(f"--- Chunk {i + 1} ---")
    print(chunk)
    print()

Found 2 relevant chunks for the test query.

--- Chunk 1 ---
--- Page 46 ---
46.
(to #10)
Do you think he lied?
#10
Now that's a stupid question. Sure
he lied.
MEDIUM SHOT - #'s 8, 10, 11, SHOOTING BETWEEN #'S 4 AND 5
#8
(to #4)
Do you?
#4
You don't have to ask me that. You
know my answer. He lied...
#8
(to #5)
Do you think he lied?
CLOSE UP - #5
He can’t answer immediately. He looks around nervously.
#5
Well... I don't know...
MEDIUM SHOT - #3, STANDING
#3
Now wait a second!
He starts to stride around table pest #'s 4, 5, 6.
#3
What are you, the kid's lawyer or
something? Who do you think you are
to start cross-examining us? Listen,
there are still eleven of us in here
who think he’s guilty.
#3 is standing behind #7 now.
#7
Right! What do you think you’re
gonna accomplish? You’re not gonna
change anybody's mind. So if you
want to be stubborn and hang this
jury go ahead. The kid'll be tried
again and found guilty sure as he's
born.
MEDIUM SHOT - #8

--- Chunk 2 ---
--- Page 10 ---
10.


## Step 6: Setting up the LLM
We will load `Qwen3-4B-Instruct-2507`. We load it in `bfloat16` precision to save memory so it fits comfortably on our free Colab GPU.

In [12]:
import torch
from transformers import pipeline

print("Loading LLM...")

model_id = "Qwen/Qwen3-4B-Instruct-2507"

# Set up the text-generation pipeline
llm_pipeline = pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"dtype": torch.bfloat16},
    device_map="auto"
)

print("LLM Loaded and ready!")

Loading LLM...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

LLM Loaded and ready!


## Step 7: The Final RAG Pipeline

You have built the retrieval engine and loaded the generator. Now, it is time to tie everything together into a complete Retrieval-Augmented Generation (RAG) pipeline.

In this final step, you will write a function that takes a user's question, fetches the most relevant information from your databases, constructs a prompt for the LLM, and generates a final answer.

**Your Tasks:**
1. **Retrieve Context:** Call your hybrid retrieval function from Step 5 to get the top `k` relevant text chunks for the user's query. Join these chunks into a single, cohesive string.
2. **Design the Prompt:** Large Language Models perform best with clear instructions. You need to construct a message history (a list of dictionaries) containing:
   * A **System Prompt:** Define the AI's persona (e.g., an expert analyzing the movie "12 Angry Men"). Crucially, instruct the AI to answer *only* using the provided context and to cite its sources if possible.
   * A **User Prompt:** Combine the user's actual question with the context string you built in step 1.
3. **Generate Answer:** Pass your message history into your `llm_pipeline`.
   * *Hint:* To keep the model grounded and prevent rambling, configure parameters like `max_new_tokens=150`, `temperature=0.1` (low temperature reduces hallucinations), and `do_sample=True`.
4. **Extract Response:** The Hugging Face pipeline returns a nested structure. You will need to extract just the text of the assistant's final generated reply.

### Expected Outcome
Your function should return a single, coherent text string containing the AI's answer. When you run the test queries at the bottom of the cell, the LLM should accurately answer the questions based exclusively on the retrieved script excerpts.

In [13]:
def ask_rag(query):
    """
    Executes the full RAG pipeline: Retrieves context, builds prompt, and generates an answer.
    """
    print(f"Question: {query}")
    print("Retrieving context...")
    contexts = hybrid_retrieve(query, top_k=3)

    # Join the retrieved chunks into a single string (e.g., separated by newlines)
    context_string = "\n\n---\n\n".join(contexts)

    # 2. Build the prompt
    # Write a strong system prompt directing the AI's behavior
    system_prompt = """You are an expert assistant analyzing the movie "12 Angry Men". ...
    Your task is to answer the user's question based STRICTLY on the provided context excerpts from the movie's script.

Rules you must follow:
1. Use ONLY the information found in the provided context. Do not use any outside knowledge about the movie.
2. If the answer cannot be found in the context, respond exactly with: "The provided context does not contain this information."
3. When possible, cite the page number where you found the answer (e.g., "as shown on Page 10").
4. Be concise and factual. Do not speculate or add commentary beyond what the context supports.
5. The jurors are referred to by number (e.g., Juror #8, #10). Preserve this notation in your answer.

"""
    # Combine the context and the user's query / Change this as needed to fit your system prompt and the way you want to present the information to the LLM.
    user_prompt = f"Context :\n{context_string}\n\nQuestion: {query}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    # 3. Generate Answer / You can play with the generation parameters to see how they affect the output. For factual answers, lower temperature and disabling sampling can help.
    outputs = llm_pipeline(
        messages,
        max_new_tokens=150,
        temperature=0.1,
        do_sample=False
    )

    # Extract just the text of the assistant's reply
    final_answer = outputs[0]["generated_text"][-1]["content"]

    return final_answer
print(ask_rag("Who had a cold?"))
print("\n" + "="*50 + "\n")
print(ask_rag("What was the murder weapon?"))
print("\n" + "="*50 + "\n")
print(ask_rag("Who was the last juror to change his vote?"))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Who had a cold?
Retrieving context...


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


#10 had a cold.  

This is stated in the context on Page 10:  
"#10 blows his nose vigorously.... I can hardly touch my nose. Know what I mean?... And how. These hot weather colds can kill you."  

Thus, Juror #10 had a cold.  

Answer: Juror #10. (as shown on Page 10)


Question: What was the murder weapon?
Retrieving context...


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The murder weapon was a knife. 

This is established when Juror #3 says, "The boy bought the knife... and three and a half hours before they found it shoved up to here in the father's chest," and when Juror #8 questions how the boy showed the knife to friends just before the murder. The context also references the "stabbing" and the "knife" being used to plunge into the father's chest. 

Thus, the murder weapon is clearly identified as a knife. 

(Note: The specific type or condition of the knife is not detailed in the provided context.)


Question: Who was the last juror to change his vote?
Retrieving context...
The last juror to change his vote was Juror #12.

This is shown on Page 134, where #12 says, "I'm changing my vote. I think he's guilty," after initially expressing uncertainty. This marks his shift from a possible acquittal to a guilty vote. There is no mention of any other juror changing their vote after this point, and Juror #3's final vote of "Not guilty" occurs later, aft

## Step 8: Baseline Comparison (LLM Without RAG)

To truly understand the value of our RAG pipeline, we need to see how the Large Language Model performs *without* it.

When you ask a standard LLM a highly specific question, it relies entirely on the knowledge it memorized during its initial training. If it doesn't know the answer, it might guess, hallucinate, or give a vague response.

**Your Task:**
Simply run the cell below. We have provided the `ask_baseline` function for you. This function asks the exact same LLM your question, but it **does not** provide any of the script excerpts from our database.

After running this, try asking both `ask_rag(query)` and `ask_baseline(query)` the same highly specific questions about the script and compare the results!

In [14]:
def ask_baseline(query):
    """
    Queries the LLM directly without any retrieved context (Baseline).
    """
    print(f"Question: {query}")
    print("Generating baseline answer...")

    # 1. Build the prompt without context
    system_prompt = "You are a helpful assistant. Answer the user's question to the best of your ability. The question will be about the movie '12 Angry men' of 1957."

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query}
    ]

    # 2. Generate Answer
    outputs = llm_pipeline(
        messages,
        max_new_tokens=150,
        temperature=0.1,
        do_sample=False
    )

    # 3. Extract just the text of the assistant's reply
    final_answer = outputs[0]["generated_text"][-1]["content"]
    return final_answer


In [15]:
#COMPARISSON WITH RAG AND NO RAG
queries = [
    "Who had a cold?",
    "What was the murder weapon?",
    "Who was the last juror to change his vote?"
]

for q in queries:
    print("=" * 60)
    print(f"QUERY: {q}")
    print("=" * 60)

    print("\n--- BASELINE (no RAG) ---")
    print(ask_baseline(q))

    print("\n--- RAG ---")
    print(ask_rag(q))

    print("\n")

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUERY: Who had a cold?

--- BASELINE (no RAG) ---
Question: Who had a cold?
Generating baseline answer...


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In the 1957 film *12 Angry Men*, there is no character explicitly mentioned as having a cold. The film is a drama centered around a jury deliberation in a capital murder case, and the focus is on the dynamics, biases, and reasoning of the 12 jurors, not on physical ailments like colds.

Therefore, based on the film's plot and dialogue, **no character is known to have a cold**. It's likely a misunderstanding or a mix-up with another film or context.

--- RAG ---
Question: Who had a cold?
Retrieving context...


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

In the 1957 film *12 Angry Men*, the murder weapon is a **knife**. 

The story centers on a jury deliberating the fate of a young man accused of murdering his father. The key piece of evidence is a knife found at the scene of the crime. However, the film focuses more on the process of jury deliberation and the characters' personal biases and emotions than on the physical details of the weapon. The knife is not shown in detail, and its significance is more about the debate over whether it was used in the murder and whether it matches the timeline of events.

It's worth noting that the original play and film do not specify the exact type of knife or its condition, so the weapon remains a

--- RAG ---
Question: What was the murder weapon?
Retrieving context...


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The murder weapon was a knife. 

This is established when Juror #3 says, "The boy bought the knife... and three and a half hours before they found it shoved up to here in the father's chest," and when Juror #8 questions how the boy showed the knife to friends just before the murder. The context also references the "stabbing" and the "knife" being used to plunge into the father's chest. 

Thus, the murder weapon is clearly identified as a knife. 

(Note: The specific type or condition of the knife is not detailed in the provided context.)


QUERY: Who was the last juror to change his vote?

--- BASELINE (no RAG) ---
Question: Who was the last juror to change his vote?
Generating baseline answer...


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In the 1957 film *12 Angry Men*, the last juror to change his vote is **Juror 8**.

Although Juror 8 initially votes "not guilty" and is the first to challenge the majority vote, he is not the last to change. The vote shifts gradually as the other jurors are persuaded through discussion and reasoning. The final vote change occurs when **Juror 10** changes his vote from "guilty" to "not guilty" after being convinced by the evidence and the discussion.

However, the **last juror to change his vote** is actually **Juror 10**, who is the final one to switch from guilty to not guilty. This happens in the final scene

--- RAG ---
Question: Who was the last juror to change his vote?
Retrieving context...
The last juror to change his vote was Juror #12.

This is shown on Page 134, where #12 says, "I'm changing my vote. I think he's guilty," after initially expressing uncertainty. This marks his shift from a possible acquittal to a guilty vote. There is no mention of any other juror changing th

## Step 9: Evaluating Our System

A crucial part of building AI systems is evaluating them. To do this, we will run both our Baseline LLM and our newly built RAG pipeline against a dataset of predefined questions about "12 Angry Men."

We will compare their answers to the `Correct Answer` provided in our dataset to see how much the retrieved context improves the model's accuracy.

**Your Task:**
Run the evaluation code block below.
* Set `PRINT_RESULTS = True` if you want to watch the models answer the questions in real-time.
* Set `PRINT_RESULTS = False` if you just want it to process everything silently in the background.

The script will automatically test the first question - you can change that variable as you wish. Once it finishes, it will save a new file called `dataset_with_results.csv` containing all the generated answers.

In [18]:
import pandas as pd
from IPython.display import display, Markdown

# ==========================================
# EVALUATION SETTINGS
# ==========================================
# Set this to False if you only want to generate the CSV without printing to the screen
PRINT_RESULTS = False

print("Loading dataset...")
# Load your local dataset
file_path = "dataset.csv"  # Ensure this file is in the same folder as your notebook
df_questions = pd.read_csv(file_path)

print(f"Loaded {len(df_questions)} questions from {file_path}")
if PRINT_RESULTS:
    display(df_questions.head())

# Define how many questions to test (e.g., first 5, 10, or len(df_questions) for all)
num_tests = len(df_questions)
# ==========================================

# 1. Initialize the new columns in the dataframe if they don't exist
if "simple_rag_answer" not in df_questions.columns:
    df_questions["simple_rag_answer"] = ""
if "baseline_answer" not in df_questions.columns:
    df_questions["baseline_answer"] = ""

# Adjust num_tests just in case the dataset is smaller than requested
num_tests = min(num_tests, len(df_questions))
print(f"\nEvaluating {num_tests} questions. This may take a few minutes...\n")

for index, row in df_questions.head(num_tests).iterrows():
    question = row["Question"]
    expected = row["Correct Answer"]

    # 2. Get answers from both systems for comparison
    baseline_model_answer = ask_baseline(question)
    rag_model_answer = ask_rag(question)

    # 3. Save the answers into the dataframe
    df_questions.at[index, "baseline_answer"] = baseline_model_answer
    df_questions.at[index, "simple_rag_answer"] = rag_model_answer

    # 4. Pretty print the results for real-time monitoring
    if PRINT_RESULTS:
        output = f"""
### Question {index + 1}: {question}

**Expected Answer:**
> {expected}

**Baseline Model Answer (No Context):**
> {baseline_model_answer}

**RAG Model Answer (With Context):**
> {rag_model_answer}

---
"""
        display(Markdown(output))

# 5. Save the updated dataframe to your local CSV
df_questions.to_csv("dataset_with_results.csv", index=False)
print(f"✅ Successfully processed {num_tests} questions and saved to 'dataset_with_results.csv'")

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Loading dataset...
Loaded 26 questions from dataset.csv

Evaluating 26 questions. This may take a few minutes...

Question: What type of train passes by the apartment right outside the window where the murder took place?
Generating baseline answer...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What type of train passes by the apartment right outside the window where the murder took place?
Retrieving context...


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Question: According to the script, what is the occupation of Juror #3?
Retrieving context...


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: How old is Juror #3 and what is his occupation?
Generating baseline answer...


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: How old is Juror #3 and what is his occupation?
Retrieving context...
✅ Successfully processed 26 questions and saved to 'dataset_with_results.csv'
